<a href="https://colab.research.google.com/github/gkestler/quasispecies_lesson_k12/blob/main/Quasispecies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Welcome to your Google Colab Notebook! 🚀

Here is the link to this notebook: https://tinyurl.com/3k9jeteh

Before we jump into the biology, here is a quick guide on how to interact with this workspace. This notebook is made of two types of blocks called **cells**: text cells (like this one) and code cells (where our simulation lives).

#### 🛠️ Essential Commands to Know:

* **How to Run a Cell:** Click inside any cell and press **`Shift + Enter`** on your keyboard (or click the **Play button ▶️** on the far left side of the code cell).
    * If you do this on a *text* cell, it will format it cleanly.
    * If you do this on a *code* cell, it will run the Python simulation! You will see a spinning loading circle next to the cell while it works, which turns into a green checkmark once it finishes.
* **How to Change Values:** To change parameters like `MUTATION_RATE`, simply click directly on the text inside the code cell, delete the old number, and type your new number.
* **Rerunning Changes:** Any time you edit a number in a code cell, you **must** press **`Shift + Enter`** (or click the Play button) again to tell the computer to update the simulation with your new settings.
* **Stuck or Need a Reset?** If a cell gets stuck, or you want to wipe the memory and start over, look at the menu bar at the top of the page and click **Runtime** -> **Restart session**.

---

### ⚠️ CRITICAL STEP 1: Boot Up the Engine!

Before you can run any of the labs or experiments below, you **MUST run the very first code cell below (Cell 1: Injected Master Engine)**.

This cell contains the hidden mathematical gears, physics loops, and graphing tools that power our virtual ecosystem. If you skip this cell or forget to run it, your interactive workbenches will throw an error and refuse to start!

*Click on the "Master Engine" code cell directly below and press **`Shift + Enter`** to turn on the simulator.* ```

In [ ]:
# --- RUN THIS CELL ONCE AT THE VERY BEGINNING TO INJECT THE MASTER ENGINE ---
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.decomposition import PCA
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML, display
from tqdm import tqdm
from PIL import Image

# Global constants shared across all exercises
GENOME_LENGTH = 50
POP_SIZE = 1000
BASELINE_FITNESS = 0.1  #The structural floor where all far-drifted mutants land safely

def init(peak_configs):
    """Generates the coordinates, target peaks, and PCA reference frames dynamically based on the fitness landscape."""
    num_peaks = len(peak_configs)
    np.random.seed(42)
    peaks = [np.random.randint(0, 4, GENOME_LENGTH) for _ in range(num_peaks)]

    # Auto-generate color palette strings
    cmap = plt.get_cmap('Set1')
    dot_colors_pool = [mpl.colors.to_hex(cmap(i)) for i in range(num_peaks)]

    # Seed sequences for reference PCA calibration space
    reference_sequences = list(peaks)
    for peak in peaks:
        for _ in range(200):
            mutant = peak.copy()
            n_mut = np.random.randint(1, 6)
            m_idx = np.random.choice(GENOME_LENGTH, size=n_mut, replace=False)
            r_shf = np.random.randint(1, 4, size=n_mut)
            mutant[m_idx] = (mutant[m_idx] + r_shf) % 4
            reference_sequences.append(mutant)

    global_pca = PCA(n_components=2)
    global_pca.fit(np.vstack(reference_sequences))
    peaks_pca = global_pca.transform(peaks)
    if num_peaks == 1:
        pltmin_max = 3.5
    else:
        pltmin_max = max([abs(peaks_pca.min()), abs(peaks_pca.max())]) * 1.6

    # 2D Grid Mapping for contour plots
    x_range = np.linspace(-pltmin_max, pltmin_max, 80)
    y_range = np.linspace(-pltmin_max, pltmin_max, 80)
    x, y = np.meshgrid(x_range, y_range)
    grid_points_discrete = np.clip(np.round(global_pca.inverse_transform(np.c_[x.ravel(), y.ravel()])), 0, 3).astype(int)
    grid_distances = [np.sum(grid_points_discrete != peak, axis=1) for peak in peaks]

    # 2D Landscape Visual Layer handles the baseline floor floor ---
    z_layers = []
    for d, config in zip(grid_distances, peak_configs):
        decay_layer = config["height"] * np.exp(-config["alpha"] * d)
        z_layers.append(np.maximum(BASELINE_FITNESS, decay_layer))

    z = np.maximum.reduce(z_layers).reshape(x.shape)

    # Return packed environment settings dictionary
    return {
        'peaks': peaks, 'global_pca': global_pca, 'peaks_pca': peaks_pca,
        'pltmin_max': pltmin_max, 'dot_colors_pool': dot_colors_pool,
        'x': x, 'y': y, 'z': z
    }

def run_simulation_data(generations, mutation_rate, peak_configs, env):
    """Executes the evolutionary model calculations strictly inside internal memory data arrays."""
    peaks = env['peaks']
    num_peaks = len(peak_configs)

    pop_segments, lineage_segments = [], []
    pop_per_peak = POP_SIZE // num_peaks
    for i, peak in enumerate(peaks):
        pop_segments.append(np.tile(peak, (pop_per_peak, 1)))
        lineage_segments.append(np.full(pop_per_peak, i, dtype=int))

    pop = np.vstack(pop_segments)
    lineages = np.concatenate(lineage_segments)

    history = {
        'populations': [], 'lineages_maps': [],
        'dist_hist': {i: [] for i in range(num_peaks)},
        'abund_hist': {i: [] for i in range(num_peaks)}
    }

    for gen in tqdm(range(generations), desc="Step 1/2: Simulating Biology", unit="gen"):
        distances = [np.sum(pop != peak, axis=1) for peak in peaks]

        # --- Hybrid selection calculation ---
        # Decay exponentially down to baseline floor, never falling below BASELINE_FITNESS
        fitnesses = [np.maximum(BASELINE_FITNESS, config["height"] * np.exp(-config["alpha"] * d))
                     for d, config in zip(distances, peak_configs)]
        fitness = np.maximum.reduce(fitnesses)

        current_pop_size = len(pop)
        selected_indices = np.random.choice(current_pop_size, size=current_pop_size, p=(fitness / np.sum(fitness)))
        pop = pop[selected_indices]
        lineages = lineages[selected_indices]

        mutation_mask = np.random.rand(len(pop), GENOME_LENGTH) < mutation_rate
        if np.any(mutation_mask):
            random_shifts = np.random.randint(1, 4, size=pop.shape)
            pop = np.where(mutation_mask, (pop + random_shifts) % 4, pop)

        for i in range(num_peaks):
            lineage_mask = (lineages == i)
            count = np.sum(lineage_mask)
            history['abund_hist'][i].append(count)
            if count > 0:
                history['dist_hist'][i].append(np.mean(np.sum(pop[lineage_mask] != peaks[i], axis=1)))
            else:
                history['dist_hist'][i].append(np.nan)

        history['populations'].append(pop.copy())
        history['lineages_maps'].append(lineages.copy())

    return history

def render_and_display_animation(generations, mutation_rate, peak_configs, history_data, env, fps=12, generate_video=True):
    """Processes graphical plotting. Switches cleanly between instant static summaries and full video controls."""
    peaks = env['peaks']
    global_pca = env['global_pca']
    peaks_pca = env['peaks_pca']
    pltmin_max = env['pltmin_max']
    dot_colors_pool = env['dot_colors_pool']
    x, y, z = env['x'], env['y'], env['z']
    num_peaks = len(peak_configs)

    fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(8, 10))
    ax3 = ax2.twinx()

    ax1.contourf(x, y, z, levels=15, cmap='viridis', alpha=0.6)
    for i, config in enumerate(peak_configs):
        ax1.scatter(peaks_pca[i, 0], peaks_pca[i, 1], c='white', marker='X', s=180, edgecolor='black', linewidth=1.5, zorder=5)
        ax1.text(peaks_pca[i, 0] + 0.5, peaks_pca[i, 1] + 0.5, config["label"], color='black', weight='bold', fontsize=9, zorder=6)
    ax1.set_xlim(-pltmin_max, pltmin_max)
    ax1.set_ylim(-pltmin_max, pltmin_max)

    scatter_pop = ax1.scatter([], [], s=20, edgecolor='k', linewidth=0.3, alpha=0.5)
    lines_dist, lines_abund = [], []
    for i, config in enumerate(peak_configs):
        line_d, = ax2.plot([], [], color=dot_colors_pool[i], linewidth=2.5, label=f'{config["label"]} (Avg Distance)')
        line_a, = ax3.plot([], [], color=dot_colors_pool[i], linestyle=':', linewidth=2, alpha=0.7, label=f'{config["label"]} (Pop Count)')
        lines_dist.append(line_d)
        lines_abund.append(line_a)

    ax2.set_title("Quasispecies Stability vs. Dispersal", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Generation")
    ax2.set_ylabel("Mean Hamming Distance from Peak", color='black')
    ax3.set_ylabel("Population Size Tracking", color='gray')
    ax2.set_xlim(0, generations)
    ax2.grid(True, linestyle='--', alpha=0.3)

    lines_all = lines_dist + lines_abund
    ax2.legend(lines_all, [l.get_label() for l in lines_all], loc='upper left', fontsize=8)
    plt.tight_layout()

    width, height = fig.canvas.get_width_height()
    target_size = (int(width * 0.7), int(height * 0.7))

    # --- FAST ANALYSIS MODE ---
    if not generate_video:
        gen = generations - 1
        pop_2d = global_pca.transform(history_data['populations'][gen])
        scatter_pop.set_offsets(pop_2d)
        scatter_pop.set_color([dot_colors_pool[lid] for lid in history_data['lineages_maps'][gen]])
        ax1.set_title(f"Final Generation {gen} Summary Plot", fontsize=11, fontweight='bold')

        x_data = np.arange(generations)
        current_max_dist, current_max_abund = 0.1, 0.1
        for i in range(num_peaks):
            lines_dist[i].set_data(x_data, history_data['dist_hist'][i])
            lines_abund[i].set_data(x_data, history_data['abund_hist'][i])
            valid_dists = [d for d in history_data['dist_hist'][i] if not np.isnan(d)]
            if valid_dists: current_max_dist = max(current_max_dist, max(valid_dists))
            if history_data['abund_hist'][i]: current_max_abund = max(current_max_abund, max(history_data['abund_hist'][i]))

        ax2.set_ylim(0, current_max_dist * 1.1)
        ax3.set_ylim(0, current_max_abund * 1.05)
        plt.show()
        return

    # --- FULL CONTROLS VIDEO PLAYER MODE ---
    frames_cache = []
    for gen in tqdm(range(generations), desc="Step 2/3: Rendering Canvas Frames", unit="frame"):
        pop = history_data['populations'][gen]
        lineages = history_data['lineages_maps'][gen]

        scatter_pop.set_offsets(global_pca.transform(pop))
        scatter_pop.set_color([dot_colors_pool[lid] for lid in lineages])
        ax1.set_title(f"Generation {gen} (Mutation Rate: {mutation_rate})", fontsize=11, fontweight='bold')

        x_data = np.arange(gen + 1)
        current_max_dist, current_max_abund = 0.1, 0.1

        for i in range(num_peaks):
            y_dist_data = history_data['dist_hist'][i][:gen + 1]
            y_abund_data = history_data['abund_hist'][i][:gen + 1]
            lines_dist[i].set_data(x_data, y_dist_data)
            lines_abund[i].set_data(x_data, y_abund_data)

            valid_dists = [d for d in y_dist_data if not np.isnan(d)]
            if valid_dists: current_max_dist = max(current_max_dist, max(valid_dists))
            if y_abund_data: current_max_abund = max(current_max_abund, max(y_abund_data))

        ax2.set_ylim(0, current_max_dist * 1.1)
        ax3.set_ylim(0, current_max_abund * 1.05)

        fig.canvas.draw()
        rgba_buffer = fig.canvas.buffer_rgba()
        frame_img = Image.frombuffer("RGBA", (width, height), rgba_buffer, "raw", "RGBA", 0, 1)
        frames_cache.append(frame_img.resize(target_size, Image.Resampling.LANCZOS))

    plt.close(fig)

    fig_play, ax_play = plt.subplots(figsize=(6, 7.5))
    ax_play.axis('off')
    artists_list = []
    for img in tqdm(frames_cache, desc="Step 3/3: Stitching Animation Player", unit="frame"):
        artists_list.append([ax_play.imshow(img, animated=True)])

    plt.tight_layout()
    mpl.rcParams['animation.embed_limit'] = 50.0
    js_anim = ArtistAnimation(fig_play, artists_list, interval=1000//fps, blit=True, repeat_delay=1000)
    display(HTML(js_anim.to_jshtml()))
    plt.close(fig_play)

# Module 1: The Single Fitness Peak & The Error Threshold

### Background
In this module, we will explore **Eigen's Paradox** and the concept of an **Error Threshold** using a quasispecies simulation framework.

A biological "quasispecies" is a cloud of genetically linked mutant variants that dynamically clusters around a master sequence. Unlike classic macroscopic organisms where individuals match a single wild-type template, highly mutable systems (like RNA viruses or early life replicators) survive as a moving cloud of mutations.

### Key Parameters to Adjust:
* `MUTATION_RATE`: The probability ($0.0$ to $1.0$) that any single sequence character will randomly mutate per generation.
* `alpha` ($\alpha$): The decay rate of the peak. A **higher** alpha means a **steeper** peak (fitness drops rapidly for even slight mutations). A **lower** alpha means a **broader, flatter** peak.

### Your Mission:
Run the simulation across various mutation rates. Find the **Critical Error Threshold**—the exact point where the mutation rate becomes too high for natural selection to hold the genetic data together, causing the population to permanently drift off the peak into random genetic noise.

### Module 1 (Part 1): Simulating random mutations in populations.

Use the cell below to configure your environmental parameters.

* To watch the full animation play over time, keep `GENERATE_ANIMATION = True`.
* Once you understand the dynamics and want to run multiple configurations quickly, set `GENERATE_ANIMATION = False`. This skips video generation and instantly displays the final structural plots, cutting down processing times by 95%!

In [ ]:
# =========================================================================
# --- EXPERIMENT WORKBENCH: SINGLE POPULATION RUN ---
# =========================================================================

# 1. ADJUST THESE PARAMETERS FOR YOUR EXPERIMENT TASK:
MUTATION_RATE = 0.005        # Try changing this! Good values might lie between 0.005 and 0.5
GENERATIONS = 60            # Keep it low for fast loops (e.g. 50-60 frames)
GENERATE_ANIMATION = True   # Set to False to disable the video and get an instant final frame summary

PEAK_CONFIGS = [
    {"height": 1.0, "alpha": 0.20, "label": "Master Sequence Peak"}
]

# =========================================================================
# --- Clean execution sequence ---
# =========================================================================
# Setup coordinates, map landscape, sample background sequences
env = init(PEAK_CONFIGS)

# Run biological step simulations
history = run_simulation_data(GENERATIONS, MUTATION_RATE, PEAK_CONFIGS, env)

# Process visualization outputs
render_and_display_animation(GENERATIONS, MUTATION_RATE, PEAK_CONFIGS, history, env, generate_video=GENERATE_ANIMATION)

## Module 1 (Part 2): Instant Parameter Sweeps & Phase Transitions

### Background
When exploring complex evolutionary systems, watching animations is helpful for building intuition, but collecting systematic data requires a parameter sweep.

By running our decoupled simulation engine across a range of values without rendering the full 2D layout, we can track how the **Final Equilibrium Hamming Distance** changes.

In physics and complex systems, a sudden jump in the state of a system is called a **Phase Transition**. In evolutionary biology, this curve shows the exact edge of the **Error Threshold**.

### Your Mission:
Run the parameter sweep below. Observe the resulting line plot:
1. At what critical mutation rate does the population completely lose its grip on the peak and collapse into a random distribution?
2. How does changing the peak sharpness (`alpha`) shift this critical breakdown point?

In [ ]:
# =========================================================================
# --- PARAMETER SWEEP WORKBENCH ---
# =========================================================================

# 1. Define the range of mutation rates to test (X-axis)
MUTATION_RATES_TO_TEST = [0.001, 0.002, 0.003, 0.005, 0.01, 0.015, 0.02, 0.025, 0.03, 0.05, 0.06, 0.07, 0.1]

# 2. Define the different landscape shapes to compare
ALPHA_CONFIGS_TO_TEST = [
    {"alpha": 1.0, "label": "Very Sharp Peak (α = 0.30)", "color": "#e41a1c"}    ]

GENERATIONS = 80  # Number of generations to let the system reach equilibrium

# =========================================================================
# --- Automated Sweep Execution (No Graphics Rendered Here) ---
# =========================================================================
results_grid = {config["label"]: [] for config in ALPHA_CONFIGS_TO_TEST}

print("Starting high-speed multi-parameter simulation sweep...")

for config in ALPHA_CONFIGS_TO_TEST:
    current_alpha = config["alpha"]
    label = config["label"]

    # Define a single-peak landscape configuration for this run
    sweep_peak_config = [{"height": 1.0, "alpha": current_alpha, "label": "Target"}]

    # Initialize the coordinate space for this specific landscape shape
    env = init(sweep_peak_config)

    for mut_rate in MUTATION_RATES_TO_TEST:
        # Run pure numerical simulation (Blazing fast because it completely skips graphics)
        history = run_simulation_data(GENERATIONS, mut_rate, sweep_peak_config, env)

        # Grab the final generation's average Hamming distance from the peak
        final_generation_distance = history['dist_hist'][0][-1]
        results_grid[label].append(final_generation_distance)

print("\nSweep complete! Generating phase transition plots...")

# =========================================================================
# --- Phase Transition Comparative Plotting Block ---
# =========================================================================
plt.figure(figsize=(9, 5.5))

for config in ALPHA_CONFIGS_TO_TEST:
    label = config["label"]
    plt.plot(
        MUTATION_RATES_TO_TEST,
        results_grid[label],
        marker='o',
        linewidth=2.5,
        markersize=7,
        color=config["color"],
        label=label
    )

# Baseline references for students to interpret
plt.axhline(y=GENOME_LENGTH * 0.75, color='gray', linestyle='--', alpha=0.7,
            label="Theoretical Random Chaos Limit (75% Mutation Dissimilarity)")

plt.title("The Error Threshold Phase Transition (Final Population Distance)", fontsize=12, fontweight='bold')
plt.xlabel("Mutation Rate (per site, per generation)", fontsize=10)
plt.ylabel(f"Mean Hamming Distance from Peak at Gen {GENERATIONS}", fontsize=10)
plt.xlim(min(MUTATION_RATES_TO_TEST)*0.5, max(MUTATION_RATES_TO_TEST) * 1.05)
plt.ylim(0, GENOME_LENGTH * 0.85)
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## Module 1 (Part 3): Shifting Landscape Shapes & Selection Thresholds

### Background
Instead of modifying the rate of genetic errors, what happens if we change the **shape of the physical world**?

By locking the `MUTATION_RATE` at a fixed value and sweeping through a wide gradient of peak steepness values (`alpha`), we can map out how selection pressure alters quasispecies cohesion.

* A low `alpha` means a flat landscape with practically no selection pressure.
* A high `alpha` means a steep landscape with aggressive selection pressure.

### Your Mission:
Run the parameter sweep below. Observe the resulting curve:
1. Explain the "U-Shape" or "V-Shape" of the transition curve. Why does the population get *closer* to the peak as alpha increases initially, but then fly *away* from the peak when alpha gets excessively high?
2. Adjust the fixed mutation rates (0.01 and 0.04) to observe different effects.
3. Find a range of alphas where selection pressure is balanced to keep the population stable.

In [ ]:
# =========================================================================
# --- LANDSCAPE SHAPE (ALPHA) SWEEP WORKBENCH ---
# =========================================================================

# 1. Define a broad range of alphas to sweep through (X-axis)
ALPHAS_TO_TEST = [0.001, 0.005, 0.01, 0.02, 0.04, 0.08, 0.15, 0.25, 0.40, 0.60, 1.0]

# 2. Test this across a couple of fixed mutation rates to see how the threshold shifts
MUTATION_CONFIGS_TO_TEST = [
    {"rate": 0.01, "label": "Low Mutation Rate (1%)", "color": "#377eb8"},
    {"rate": 0.04, "label": "Moderate Mutation Rate (4%)", "color": "#e41a1c"}
]

GENERATIONS = 80  # Generations allowed for the population to reach equilibrium

# =========================================================================
# --- Automated Sweep Execution (Skipping Graphics Rendering) ---
# =========================================================================
results_grid = {config["label"]: [] for config in MUTATION_CONFIGS_TO_TEST}

print("Starting high-speed landscape-shape simulation sweep...")

for config in MUTATION_CONFIGS_TO_TEST:
    fixed_mut_rate = config["rate"]
    label = config["label"]

    for alpha_val in ALPHAS_TO_TEST:
        # Define a single-peak landscape configuration using the current alpha value
        sweep_peak_config = [{"height": 1.0, "alpha": alpha_val, "label": "Target"}]

        # Initialize the unique coordinate projection space for this specific alpha curve
        env = init(sweep_peak_config)

        # Run pure numerical simulation
        history = run_simulation_data(GENERATIONS, fixed_mut_rate, sweep_peak_config, env)

        # Record the final generation's average Hamming distance from the peak
        final_generation_distance = history['dist_hist'][0][-1]
        results_grid[label].append(final_generation_distance)

print("\nSweep complete! Generating selection threshold plots...")

# =========================================================================
# --- Plotting the Selection Threshold Jump ---
# =========================================================================
plt.figure(figsize=(9, 5.5))

for config in MUTATION_CONFIGS_TO_TEST:
    label = config["label"]
    plt.plot(
        ALPHAS_TO_TEST,
        results_grid[label],
        marker='s',
        linewidth=2.5,
        markersize=6,
        color=config["color"],
        label=label
    )

# Baseline references for student analysis
plt.axhline(y=GENOME_LENGTH * 0.75, color='gray', linestyle='--', alpha=0.7,
            label="Theoretical Random Chaos Limit (75% Dissimilarity)")

plt.title("Selection Landscape Thresholds (Final Population Closeness)", fontsize=12, fontweight='bold')
plt.xlabel("Landscape Decay Rate / Peak Sharpness (alpha)", fontsize=10)
plt.ylabel(f"Mean Hamming Distance from Peak at Gen {GENERATIONS}", fontsize=10)
# plt.xscale('log')  # Log scale highlights the distinct jump behavior across magnitudes
plt.ylim(0, GENOME_LENGTH * 0.85)
plt.grid(True, which="both", linestyle='--', alpha=0.4)
plt.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

## Module 1: Discussion Questions

Now that you have executed both individual mutation runs and parameter sweeps, analyze the data trends in your plots to answer the following questions.

> **[Question 1.1]** Look at your mutation rate sweep plot. When the landscape decay rate ($\alpha$) is set extremely high, describe the exact shape of the population's equilibrium curve. Why does it remain flat near zero before suddenly jumping vertically up toward the chaos threshold?
>
> **[Question 1.2]** When running the simulation under a moderate mutation rate ($4\%$), look at how the population behaves as you sweep through different landscape sharpness ($\alpha$) values. Why does the population completely lose its grip on the peak at both the very flat end ($\alpha \le 0.1$) and the very sharp end ($\alpha \ge 1.0$), but manages to find a stable "window of viability" in the middle?
>
> **[Question 1.3]** Refer back to the blue line representing a low mutation rate ($1\%$) on your $\alpha$-sweep plot. When the landscape is completely flat ($\alpha = 0$), the population stabilizes at a mean Hamming distance of roughly 24 rather than the maximum theoretical chaos limit of 37.5. Explain why the population has not reached full randomization by generation 80.
>
> **[Question 1.4]** Based on the mathematical definition of Eigen's error threshold, if you keep the mutation rate locked at $4\%$ but double the length of the genome from 50 to 100, what would you expect to happen to the population's ability to hold onto a sharp peak? Would the "cliff edge" shift to the left or to the right?

# Module 2: The Battle of Evolutionary Strategies (Survival of the Flattest)

In Module 1, we explored a harsh world dominated by a single, sharp genetic peak. We discovered **Eigen's Error Threshold**: if a species mutates too rapidly, it can no longer preserve its genetic identity and drops off its peak into chaos.

But what happens when different biological lineages with completely different genetic designs go head-to-head in the exact same environment?

In this module, we introduce a classic evolutionary showdown: **The Fittest vs. The Flattest**.

### The Competitors
* **Lineage A (The Fittest):** This species occupies an ultra-fit, towering peak ($\text{Height} = 1.0$). It replicates incredibly fast when it is perfect, but its peak is a razor-thin needle ($\alpha = 3.0$). Even a single nucleotide error causes its fitness to plummet straight down to the background floor.
* **Lineage B (The Flattest):** This species occupies a much lower, humble plateau ($\text{Height} = 0.55$). It reproduces much more slowly than Lineage A under ideal conditions. However, its peak is exceptionally broad ($\alpha = 0.15$). If it accumulates a few mutations, its fitness barely drops at all, cushioning its descendants from mutational harm.

## Module 2 (Part 1)
1. **Test 1 (Low Mutation Environment):** Set `MUTATION_RATE = 0.01` in the code cell below and run the simulation.
2. **Test 2 (High Mutation Environment):** Increase the parameter to `MUTATION_RATE = 0.04` and rerun the simulation.
3. **Observe the Outputs:** Pay close attention to the **Population Size Tracking (Dotted Lines)** in the bottom plot. Because the total population size is capped at $1000$, one lineage growing means the other is actively being driven to extinction.
    * Watch the 2D map in the animation to see how the population physically shifts locations over generations.
4. **Explore:** Add more populations with various fitness curves. How do they collapse? How do they separate?

In [ ]:
# =========================================================================
# --- MODULE 2: MULTI-PEAK COMPETITION WORKBENCH ---
# =========================================================================

MUTATION_RATE = 0.01        # Try different mutation rates!
GENERATIONS = 100
GENERATE_ANIMATION = True

# --- Define the Competitors ---
# We introduce two distinct genetic strategies competing in the same world:
PEAK_CONFIGS = [
    {
        "height": 1.0,   # High absolute fitness peak
        "alpha": 3.0,    # Extremely sharp (Any mutation causes a catastrophic drop)
        "label": "Lineage A: The Fittest (Sharp Peak)"
    },
    {
        "height": 0.55,   # Much lower absolute fitness peak
        "alpha": 0.15,   # Very broad (Mutants retain high fitness nearby)
        "label": "Lineage B: The Flattest (Broad Hill)"
    }
]

# Run the unified multi-peak landscape engine
env = init(PEAK_CONFIGS)
history = run_simulation_data(GENERATIONS, MUTATION_RATE, PEAK_CONFIGS, env)
render_and_display_animation(GENERATIONS, MUTATION_RATE, PEAK_CONFIGS, history, env, generate_video=GENERATE_ANIMATION)

## Module 2: Discussion Questions

> **[Question 2.1]** When the mutation rate is low ($1\%$), which lineage wins the competition? Explain why this outcome aligns with the traditional phrase *"Survival of the Fittest"*.
>
> **[Question 2.2]** When you crank the mutation rate up to $4\%$, something highly counter-intuitive happens. Describe the population shift you observe. Why is the "fitter" master sequence driven to complete extinction by a lower-stature competitor?
>
> **[Question 2.3]** Look at the **Mean Hamming Distance** lines for both lineages during the high mutation ($4\%$) run. Why does Lineage A completely lose control of its peak, while Lineage B manages to stay stably localized on its broad hill?
>
> **[Question 2.4]** Based on what you have observed in Modules 1 and 2, explain why a virus with an incredibly high mutation rate (like HIV or Influenza) might evolve to have a flat, smooth fitness landscape rather than a single, high, ultra-specialized peak.